In [1]:
import torch.optim as optim

from src.training import get_accuracy, OneHotEncode
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    #target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    #target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [3]:
from src.mnist_model import MNIST_CNN

# Instantiate the model
model = MNIST_CNN().to(device)
model.set_scramble_distance(0.05)
#criterion = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [4]:
# Training loop
num_epochs: int = 500
target_accuracy: float = .96
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.layer2.scale.data.clamp_(min=1.0)
        model.layer3.scale.data.clamp_(min=1.0)

    # Assess progress:
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    model.binary_mode()
    bin_validation_accuracy: float = get_accuracy(model, val_dataloader)
    model.scramble_mode()

    if validation_accuracy > target_accuracy:
        model.set_scramble_distance(min(model.scramble_distance + .1, maximum_scramble_distance))
        print(f"Scramble distance: {model.scramble_distance:.2f}")
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}, Bin Accuracy: {bin_validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.0988, Bin Accuracy: 0.1014
Epoch [2/500], Accuracy: 0.9475, Bin Accuracy: 0.6703
Epoch [3/500], Accuracy: 0.9575, Bin Accuracy: 0.6669
Scramble distance: 0.15
Epoch [4/500], Accuracy: 0.9667, Bin Accuracy: 0.6068
Scramble distance: 0.25
Epoch [5/500], Accuracy: 0.9624, Bin Accuracy: 0.6929
Scramble distance: 0.35
Epoch [6/500], Accuracy: 0.9601, Bin Accuracy: 0.6966
Epoch [7/500], Accuracy: 0.9544, Bin Accuracy: 0.7352
Epoch [8/500], Accuracy: 0.9554, Bin Accuracy: 0.7093
Epoch [9/500], Accuracy: 0.9584, Bin Accuracy: 0.7331
Scramble distance: 0.45
Epoch [10/500], Accuracy: 0.9618, Bin Accuracy: 0.6718
Epoch [11/500], Accuracy: 0.9574, Bin Accuracy: 0.7467
Epoch [12/500], Accuracy: 0.9595, Bin Accuracy: 0.7171
Epoch [13/500], Accuracy: 0.9591, Bin Accuracy: 0.7128
Scramble distance: 0.55
Epoch [14/500], Accuracy: 0.9658, Bin Accuracy: 0.7117
Epoch [15/500], Accuracy: 0.9566, Bin Accuracy: 0.7512
Epoch [16/500], Accuracy: 0.9568, Bin Accuracy: 0.7438
Scramble 

KeyboardInterrupt: 

In [ ]:
from src.improved_model import BinarizingNetwork


def get_signed_accuracy(model: BinarizingNetwork, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy

test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]

print(model.layer3.bias)
print(get_signed_accuracy(model, val_dataloader))


In [5]:
torch.save(model.state_dict(), "deeply_trained_conv_v2.pth")